# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jahnzaibakhtar/Flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
Signal check 1 — staleness behind the refresh flag: older content (high days_since_update) should
show more decline signal than fresh content. I check this against is_declining_label as a
diagnostic only, never as a feature (label trap).

Signal check 2 — CTR-vs-position: content with CTR below what's typical for its position band is
underperforming relative to peers, independent of absolute position.

[Fill in after running the code cell below: state each verdict — CONFIRMED / OPPOSITE / MIXED /
FALSE — citing the actual bucket numbers you see.]

My rule, in plain words: a content item is worth reviewing if it's stale (hasn't been updated in
180+ days) AND its CTR is below the median for other content at a similar search position. Both
conditions must hold — staleness alone doesn't mean decline (see signal check 1), so I require
the CTR gap too.

Reason codes this rule can output:
- stale_and_ctr_below_position_peers — both conditions true, item is scored and queued.
- no_action — either condition false, or no position data available to compare against.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Signal check 1: staleness vs decline label (diagnostic only, not a feature)
df["staleness_bucket"] = pd.cut(
    df["days_since_update"],
    bins=[0, 90, 180, 365, df["days_since_update"].max()],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)
staleness_table = df.groupby("staleness_bucket").agg(
    n=("content_id", "count"),
    decline_rate=("is_declining_label", "mean")
)
print("Staleness vs decline rate:")
print(staleness_table)

# Signal check 2: CTR vs position band
df["position_bucket"] = pd.cut(
    df["avg_position"].replace(0, pd.NA),  # 0 = no data, not rank 0
    bins=[0, 3, 10, 20, df["avg_position"].max()],
    labels=["top3", "4-10", "11-20", "20+"]
)
ctr_table = df.groupby("position_bucket").agg(
    n=("content_id", "count"),
    median_ctr=("ctr", "median")
)
print("\nCTR by position band:")
print(ctr_table)

df["ctr_vs_position_gap"] = df.groupby("position_bucket")["ctr"].transform(
    lambda s: s.median() - s
)
print(f"\nRows with usable position data: {df['position_bucket'].notna().sum()} / {len(df)}")

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
Score: stale AND underperforming AND has position data, weighted by the size of the CTR gap
(readable on purpose, no fitted weights). Rank descending, write the full queue to CSV.

In [ ]:
stale = (df["days_since_update"] >= 180).astype(int)
underperforming = (df["ctr_vs_position_gap"] > 0).astype(int)
has_position_data = df["position_bucket"].notna().astype(int)

df["score"] = stale * underperforming * has_position_data * df["ctr_vs_position_gap"]
df["reason_code"] = df["score"].apply(
    lambda s: "stale_and_ctr_below_position_peers" if s > 0 else "no_action"
)
df["action"] = df["score"].apply(lambda s: "review_for_refresh" if s > 0 else "no_action")

queue = df.sort_values("score", ascending=False)[
    ["content_id", "client_id", "score", "reason_code", "action",
     "days_since_update", "avg_position", "ctr", "ctr_vs_position_gap"]
]

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Queue written: {(df['score'] > 0).sum()} items flagged for review out of {len(df)} total")

base_rate = df["is_declining_label"].mean()
print(f"Base rate of is_declining_label: {base_rate:.1%}")

import numpy as np
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

for k in [10, 20, 50]:
    p = precision_at_k(df["score"].values, df["is_declining_label"].values, k)
    print(f"precision@{k}: {p:.2f} (base rate: {base_rate:.2f})")

# save the run's metrics receipt
import json
metrics = {
    "n_flagged": int((df["score"] > 0).sum()),
    "n_total": len(df),
    "base_rate": float(base_rate),
    "precision_at_10": float(precision_at_k(df["score"].values, df["is_declining_label"].values, 10)),
    "precision_at_20": float(precision_at_k(df["score"].values, df["is_declining_label"].values, 20)),
    "precision_at_50": float(precision_at_k(df["score"].values, df["is_declining_label"].values, 50)),
}
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
Code cell above prints top20 = queue.head(20) — for each row, one line:

1. content_id [X] — action: review_for_refresh. Reason: stale_and_ctr_below_position_peers
   ([N] days since update, CTR [Z]% vs position-band median). Confidence: [high/medium] because
   [state actual gap size]. What would make it wrong: if this is a seasonal or evergreen-by-design
   page not meant to rank right now, staleness reads as decline but isn't.
2-20. [same format, filled from the real printed rows — vary the "what would make it wrong" line
   per item's actual position_bucket and gap size, don't copy-paste the same caveat 20 times]

In [ ]:
top20 = queue[queue["score"] > 0].head(20)
top20

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
Weak picks: [fill in after inspecting top20 — e.g. "item at rank 15 sits in the 20+ position
bucket, where CTR is naturally low for everyone, so its 'gap' may reflect the bucket's coarseness
rather than real underperformance relative to true peers." A weak pick found here is expected —
per building-baselines, finding at least one in a top-20 review means the review did its job.]

Leakage check: the score uses only days_since_update, avg_position, ctr, and their derived
buckets/gaps — none of these are label-derived. is_declining_label, trend_direction, and
trend_pct appear only in the evaluation step (precision@K, the signal-check bucket tables), never
inside the score formula itself. No product-decision flags or future-window fields were used —
everything here is trailing, same-period data from the starter CSV.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.